IMPORT IMPORTANT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

LOAD THE DATASET

In [3]:
df = pd.read_csv("/content/drive/MyDrive/DatasetAuspify.csv")
print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [4]:
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [5]:
df.shape

(8790, 10)

 SELECTING RELEVANT DATASET FEATURES

In [6]:
# 1. Define the input features
features = [
    "release_year",
    "rating",
    "country",
    "listed_in",
    "director"
]

# 2. Select input features (X)
X = df[features].copy()

# 3. Select target variable (y)
y = df["type"].copy()

# 4. Inspect the selected data
print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nSelected Features:")
display(X.head())

print("\nTarget Distribution:")
print(y.value_counts())

print("\nMissing Values:")
print(X.isnull().sum())


Input Features Shape: (8790, 5)
Target Shape: (8790,)

Selected Features:


,release_year,rating,country,listed_in,director
0,2020,PG-13,United States,Documentaries,Kirsten Johnson
1,2021,TV-MA,France,"Crime TV Shows, International TV Shows, TV Act...",Julien Leclercq
2,2021,TV-MA,United States,"TV Dramas, TV Horror, TV Mysteries",Mike Flanagan
3,2021,TV-PG,Brazil,"Children & Family Movies, Comedies",Bruno Garotti
4,1993,TV-MA,United States,"Dramas, Independent Movies, International Movies",Haile Gerima



Target Distribution:
type
Movie      6126
TV Show    2664
Name: count, dtype: int64

Missing Values:
release_year    0
rating          0
country         0
listed_in       0
director        0
dtype: int64


 ENCODING CATEGORICAL VARIABLES

In [7]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Define categorical and numerical features
categorical_features = [
    "rating",
    "country",
    "listed_in",
    "director"
]

numerical_features = [
    "release_year"
]

# 2. Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Create the encoding transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         categorical_features),

        ("numerical",
         "passthrough",
         numerical_features)
    ]
)

# 4. Fit and transform training data
X_train_encoded = preprocessor.fit_transform(X_train)

# 5. Transform testing data
X_test_encoded = preprocessor.transform(X_test)

# 6. Inspect encoded data
print("Training Data Shape:", X_train_encoded.shape)
print("Testing Data Shape:", X_test_encoded.shape)

print("\nTraining Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)


Training Data Shape: (7032, 4354)
Testing Data Shape: (1758, 4354)

Training Target Shape: (7032,)
Testing Target Shape: (1758,)


 TRAIN CLASSIFICATION MODELS

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Create classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

# 2. Train each model
for name, model in models.items():
    model.fit(X_train_encoded, y_train)
    print(name, "trained successfully.")

Logistic Regression trained successfully.
Decision Tree trained successfully.
Random Forest trained successfully.


EVALUATE PREDICTION PERFORMANCE


In [9]:
from sklearn.metrics import accuracy_score, classification_report

# Evaluate each trained model
for name, model in models.items():

    # Make predictions using testing data
    y_pred = model.predict(X_test_encoded)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Display evaluation results
    print("\nModel:", name)
    print("Accuracy:", round(accuracy, 4))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Model: Logistic Regression
Accuracy: 0.9767

Classification Report:
              precision    recall  f1-score   support

       Movie       0.98      0.99      0.98      1225
     TV Show       0.97      0.95      0.96       533

    accuracy                           0.98      1758
   macro avg       0.98      0.97      0.97      1758
weighted avg       0.98      0.98      0.98      1758


Model: Decision Tree
Accuracy: 0.9886

Classification Report:
              precision    recall  f1-score   support

       Movie       0.99      1.00      0.99      1225
     TV Show       0.99      0.97      0.98       533

    accuracy                           0.99      1758
   macro avg       0.99      0.98      0.99      1758
weighted avg       0.99      0.99      0.99      1758


Model: Random Forest
Accuracy: 0.9829

Classification Report:
              precision    recall  f1-score   support

       Movie       0.98      0.99      0.99      1225
     TV Show       0.98      0.96      0.9

COMPARE MODEL ACCURACY

In [10]:
import pandas as pd

# 1. Store model accuracy scores
results = []

for name, model in models.items():
    y_pred = model.predict(X_test_encoded)

    accuracy = accuracy_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy
    })

# 2. Create comparison table
comparison_df = pd.DataFrame(results)

# 3. Sort models by accuracy
comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

# 4. Display comparison results
display(comparison_df)

# 5. Identify the best model
best_model = comparison_df.iloc[0]

print("Best Model:", best_model["Model"])
print("Best Accuracy:", round(best_model["Accuracy"] * 100, 2), "%")

,Model,Accuracy
1,Decision Tree,0.988623
2,Random Forest,0.982935
0,Logistic Regression,0.976678


Best Model: Decision Tree
Best Accuracy: 98.86 %
